### ????? ???? ???????

??????????, ??? ???????? ????-??????????? ? ???????? Booking. ???? ?? ??????? ???????? — ??? ????????? ?????, ??????? ??????????? ???? ???????. ????? ?? ???????? ??????????? ????? ?????? ???????? ?????????? ??????, ??????? ????????????? ??????? ?????. ???? ???????????? ?????? ?????? ?????????? ?? ???????????? ??????????, ??, ????????, ????? ????? ???? ????????, ? ??? ????? ?????????.

In [ ]:
%%capture

# ????????? ?????????????? ?????????:
!pip install Afinn
!pip install nltk[twitter] 

# ??????????? ??????? ? ????????? ??????
import pandas as pd
import numpy as np
import re
import geopy.distance
from scipy import stats
from category_encoders import  OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import f_classif

# ??????????? ??? ???????????????? ???????:
import string
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob, Blobber
from textblob.sentiments import NaiveBayesAnalyzer
from afinn import Afinn
nltk.download('vader_lexicon')

# ??????????? ??? ????????????:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# ????????? ??????????? ??????????? ?? ??????:
from IPython.display import display
from IPython.display import Image
from termcolor import colored
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', 200)

# ????????? ???????????
!ls ../input/projectdata/img_score.png
!ls ../input/projectdata/img.png
!ls ../input/projectdata/wait.png

# ??????????? ??? ?????????? ??????:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor # ?????????? ??? ???????? ? ???????? ??????
from sklearn import metrics # ??????????? ??? ?????? ???????? ??????

# ????????? RANDOM_SEED ? ?????? ???????, ????? ???????????? ???? ??????????????:
RANDOM_SEED = 42
!pip freeze > requirements.tx

In [ ]:
# ?????? ?? ????????????
DATA_DIR = '/kaggle/input/sf-booking/'
df_train = pd.read_csv(DATA_DIR+'/hotels_train.csv') 
df_test = pd.read_csv(DATA_DIR+'hotels_test.csv') 
sample_submission = pd.read_csv(DATA_DIR+'/submission.csv')

In [ ]:
# ?????????? ????? ? ???? ? ???? ???????
df_train['sample'] = 1 # ???????? ?????
df_test['sample'] = 0 # ???????? ????

# ? ????? ? ??? ??? ???????? reviewer_score, ????????? ??????:
df_test['reviewer_score'] = 0 

In [ ]:
Image('../input/projectdata/img.png')

# 0. ???????? ??????:

In [ ]:
hotels = df_test.append(df_train, sort=False).reset_index(drop=True)
display(hotels.info())

hotels_isnull = hotels.isnull().sum()
print(
    f'\n??????? ? ?????? ?????????? ??????:\n{hotels_isnull[hotels_isnull > 0]}'
)

# 1. ???????????? ? ?????????????? ????????? (FEATURE ENGINEERING):

## 1.1 ?????? ????????? ?????????????? ?????
- ???????? ? ????? ????? ('hotel_address', 'hotel_name')
- ?????????? ????? - ?????????????? ?????? ? ??????? ????? ('lat', 'lng')
- ??????? ???? ????? ?? ????????? ??? ('average_score')
- ????? ?????????? ??????? ?? ????? ('total_number_of_reviews')
- ????? ?????????? ?????? ??? ??????? ('additional_number_of_scoring')

### <span style="color:blue">???????? ????? ('hotel_name')</span>

In [ ]:
# ????????? ?????? ? ???????? ?? ??????? ????????? ? ??????????:
pivot_n = hotels[['hotel_address', 'hotel_name']].groupby(
    by='hotel_name'
).nunique()

name_rep = pivot_n[pivot_n['hotel_address'] != 1].index.to_list()
mask = hotels['hotel_name'].isin(name_rep)
addresses_rep = hotels[mask][['hotel_name', 'hotel_address']].value_counts()

print(colored('???????????? ???????? ?????:', 'green'))
display(addresses_rep)

# ????? ????????? ??? ????? ??? ??????? - ??????? ? ???????? ?????:
hotels['hotel_name'] = hotels.apply(
    lambda row: (row['hotel_name'] + ' ' + row['hotel_address'].split()[-2])
    if (row['hotel_name'] in name_rep)
    else row['hotel_name'], axis=1)

print(colored('\n???????? ?????????? ?????????:', 'green'))
mask = hotels['hotel_name'].str.contains(name_rep[0])
display(hotels[mask]['hotel_name'].value_counts())

    ???????? ????? ????? ???????????? ??? ???????? ??????, ?????? ???? ??? ??????????. ??? ???????? ??? ?? ???? ???? ?? ????? ???-?? ????????. ?????? ?????? ?????? ? ???????? ??? ?????? ???????? ?????? ??-?? ????? ???????????, ??????? ?? ????? ??????? ????????? ????????.

    ????????? ???????? ?????? ???? ?????-?? ??????????? ??????????. ???????? - ??? ?????????? ?????????????? ?????. ????? ????? ?????????????, ????????, ?? ???????????? (???-?? ???????) ??? ?? ???????? (??????? ???). ?? ????? ?????????, ??? ??? ??????? ????? ??????????? ?? ?????? ??? ??????? ? ??????????? ???????, ??? ??????? ?? ??????? ???????.

    ? ?????? ????????? ? ???? ???????? ????? ???????, ?? ?????, ?? ????? ??????? ???? ????????? ? ?? ??????? ?? ??????? ??????????.

    ? ???? ?????????? ????????, ????????? ??? ??? ??????????????. ????? ? ??? ????? ?????? ?????? ?? ??????? ? ?????????? ?????????? ????? (??? ???????).

In [ ]:
# ??????? ??? ?????? ?? ????????, ? ??????? ? 100 ???????:
hotels_names = hotels.groupby('hotel_name')['average_score'].agg(
    ['mean', 'count'])

mask = (hotels_names['mean'] >= 9.0) & (hotels_names['count'] >= 100)
hotels_top = hotels_names[mask]

print(colored(
    '\n?????? ?? ??????? ?????, ??????? ??????? 100 ???????:', 'green'))

display(hotels_top)

hotels['hotel_cat'] = hotels['hotel_name'].apply(
    lambda x: x if x in hotels_top.index else 'other')

# ???????? ?????????? ???????????:
ord_encoder = OrdinalEncoder()
hotels['hotel_cat'] = hotels['hotel_cat'].astype('category')
hotels_bin = ord_encoder.fit_transform(hotels['hotel_cat'])
hotels = pd.concat([hotels, hotels_bin], axis=1)

# ?????? ???????? ???????
col = hotels.columns.tolist()
col[19] = 'hotel_code'

hotels.set_axis(col, axis=1, inplace=True)
hotels.drop('hotel_cat', axis=1, inplace=True);

### <span style="color:blue">????? ????? ('hotel_address')</span>

In [ ]:
addresses = pd.Series(hotels['hotel_address'].unique())

print(colored(
    '\n?????????? ??????????? ????????? ????? ? ??????? ??????:', 'green'))
display(addresses)

    ?? ?????? ????? ??????? ??????? ?????? ???????? ??????????, ??? ?? ????????. ??????, ????? ? ?????? ??? ?????????????? ?????? - ????? ????? ??????????????? ??????????!
    
    ???????????????? ????? ?????? ????? ?? ??????? ? ?????? ?????? ?????. ????? ?????? ???????? ?????? ???????? ?? ??????????? ??????? ?? ?????. ????? ?????? ??????????, ?????? ???????, ??????????? ????? ?? Booking.com, ????? ????????? ????????????????? ?????.

In [ ]:
# ??? ??????????? ???-?????????? ????? ??????????????? locationtagger, 
# ?? ?? ???????? ???????? ???????? ? ? ????????
# ????????? ? ??? ????????? ?????????? ?????????? ????? ? ??????? -
# ????????????? ??????? ?????????? ??????:
countries = addresses.apply(lambda x: x.split()[-1])
countries = countries.apply(
    lambda x: 'United Kingdom' if x == 'Kingdom' else x)

cities = addresses.apply(lambda x: x.split()[-2])
cities = cities.apply(lambda x: 'London' if x == 'United' else x)

addresses = pd.DataFrame(
    {'address': addresses, 'country': countries, 'city': cities})

print(colored(
    '\n? ?????? ?????? ????? ???????????? ?????? ?? ?????? ??????:', 'green'))
display(addresses.groupby(['country', 'city']).count())

# ????? ???????? ? ?????? ???-?? ???? - ? ?????? ?????? 
#(??? ????? ????? ??????? ???????????? ? ??????????????? ??????????):
hotels['hotel_country'] = hotels['hotel_address'].apply(
    lambda x: 'United Kingdom' if x.split()[-1] == 'Kingdom' else x.split()[-1])

# ???????? ?????????? ? ??????:
hotels['hotel_country'] = hotels['hotel_country'].astype('category')
hotels['hotel_country_code'] = hotels['hotel_country'].cat.codes

In [ ]:
def get_postcode(address):
    """?????????? ????????? ??????? ?? ?????? ? ???????."""
    address_list = address.split()
    postcode_place = {
        'Kingdom': -4, 
        'Spain': -3, 
        'France': -3, 
        'Austria': -3, 
        'Netherlands': -4, 
        'Italy': -3
    }
    postcode = str(address_list[postcode_place[address_list[-1]]])
    
    if address_list[-1] == 'Kingdom' and postcode[-1].isalpha():
        return postcode[:-1]
    else:
        return postcode

    
# ??????? ???????? ?????? ????? ??? ????????????? ??????:
hotels['hotel_postcode'] = hotels['hotel_address'].apply(get_postcode)


"""
??????, ??? ??????? ?????? ?? ??? - ?????/???????????? ?????/????????? -
???????? ????????, ??????? ????????????? ??? ? ???? ?? ?????. ??? ??? 
??????? ? ?????? ?????? ?? ?????, ? ????????? ???? ??????? ??????????????,
????????? ?????????? ??? ???????? ?? ???????? ??????????.

?????? ??? ? ??? ? ????? ?????? ?????? ??? ????????; ? ???? ????? ?????? ?
??????????. ? ? ??? ?? ????, ??? ?????????????? ????????? ???? ???????
?? ???????? ?? ???????? ???????. ? ???? ...
"""

# ?? ????? ? ??????? ? ???? ??????? - ????????? ????????? ??????:
districts = pd.read_csv('../input/projectdata/districts.csv')
districts['PostCode'] = districts['PostCode'].apply(lambda x: str(x))
districts = districts.set_index('PostCode')
districts = districts.to_dict()
districts = districts['Type of District']

hotels['district_type'] = hotels['hotel_postcode'].apply(lambda x: districts[x])

# ???????? ??? ??????, ???????? ???????? ????????????? ?????????????????
# (???????????? ????? - ????? ?????? ???????, ???????? - ????????):
def get_district_code(district_type, country_code):
    """?????????? ???? ?????? ? ??????????? ?? ????????? ? ??????."""
    if district_type == 'History Center':
        return country_code
    elif district_type == 'Center':
        return country_code  + 6
    elif district_type == 'Other':
        return country_code + 12
    else:
        return country_code + 18

    
hotels['district_code'] = hotels.apply(
    lambda row: get_district_code(row['district_type'], row['hotel_country_code']
), axis=1)


# ????????? ?????? ???????? ? ??????????? ?? ???????????? ?????? ? ??? ? ????????
# ??? ????? ??????????? ?????????? ? ??????? ???????? ????? ? ??????????? ?? ???????:
pivot_adr = pd.pivot_table(
    hotels, 
    index='hotel_postcode', 
    values='average_score', 
    aggfunc='mean'
)
pivot_adr = pivot_adr.reindex(
    pivot_adr['average_score'].sort_values(ascending=False).index
)

mm_scaler = MinMaxScaler()
x_data = pivot_adr.values
x_scaled = mm_scaler.fit_transform(x_data)
pivot_adr['district_rating'] = x_scaled

hotels['district_rating'] = hotels['hotel_postcode'].apply(
    lambda x: pivot_adr['district_rating'][x]
)

hotels[['hotel_address', 
        'hotel_postcode', 
        'district_rating', 
        'district_type', 
        'hotel_country_code', 
        'district_code']][:10]

#### <span style="color:green">????????????? ??????????? ??????? ???????? ??????? ?????????? ? ??????????? ?? ?????? ???????????? ?????:</span>

In [ ]:
pivot_district_type = hotels[hotels['sample'] == 1].groupby(
    ['hotel_country', 'district_type'])['reviewer_score'].mean().unstack()

cities_rus = ['????', '?????', '?????', '?????????', '?????????', '??????']

fig1, ax1 = plt.subplots(figsize=(15, 6))
sns.set_style('whitegrid')
district_type_plot = pivot_district_type.plot(
    ax=ax1, 
    kind='bar', 
    width=0.85, 
    color=['indianred', 'darksalmon', 'cadetblue', 'slateblue'],
    label=cities_rus, 
    rot=0)

ax1.set_title('??????????? ??????? ??????\n?? ???????????? ?????', 
              pad = 20, size=16)
ax1.set_xlabel('')
ax1.set_xticklabels(cities_rus, fontsize=16)
ax1.legend(['?????', '???????????? ?????', '????????? ??????', '????????'], 
           loc='upper left', fontsize=12)

# ??? ??????????? ??????? ?? ????????? ??????? ??????:
for p in district_type_plot.patches:
    district_type_plot.annotate(
        '{:.1f}'.format(p.get_height()), (p.get_x()+0.1, p.get_height()),
    ha='center', va='bottom', fontsize=14)

plt.ylim(7, 10);

### <span style="color:blue">?????????????? ?????????? ????? ('lat', 'lng')</span>

??????, ?????????? ???? ?? ???? ???? ??? ????? ???????? ??????, ?? ????? ??????????? ??????? ?? ??? ?????-?? ???????? ??????????. ????????, ?????????? ?? ?????? ??????. ?? ?????????? ? ???????? ?? ??????, ??? ?????? ? ???? ???????? ???? ????????. ??????? ?????? ????????, ????? ??????? ????? ???????.

In [ ]:
without_coor = len(hotels[hotels.isnull().any(1)]['hotel_name'].value_counts())
print(f'?????????? ?????? ? ?????????????? ????????????: {without_coor}')

# ????????? ??????? ???????? ????????? ??? ??????? ??????:
pivot_coor = pd.pivot_table(
    hotels, 
    index='hotel_postcode', 
    values=['lat', 'lng'], 
    dropna=True, 
    aggfunc='mean')

hotels['lat'] = hotels.apply(
    lambda row: pivot_coor['lat'][row['hotel_postcode']] \
    if np.isnan(row['lat']) else row['lat'], axis=1)
hotels['lng'] = hotels.apply(
    lambda row: pivot_coor['lng'][row['hotel_postcode']] \
    if np.isnan(row['lng']) else row['lng'], axis=1)

In [ ]:
# ??? ???????????? ?????? ???????? ????? ?????????? ?????? ??????? ?????? 
# (? ?????: https://dateandtime.info/):
centers = pd.DataFrame(
    data = [
        ['United Kingdom', 'London', 51.5085300, -0.1257400],
        ['France', 'Paris', 48.8534100, 2.3488000],
        ['Spain', 'Barcelona', 41.3887900, 2.1589900],
        ['Italy', 'Milan', 45.4642700, 9.1895100],
        ['Austria', 'Vienna', 48.2084900, 16.3720800],
        ['Netherlands', 'Amsterdam', 52.3740300, 4.8896900],
    ],
    columns= ['country', 'city', 'center_lat', 'center_lng']
).set_index('country')


def get_dist_to_center(country, lng, lat):
    """?????????? ????????? ?? ?????? ?????? ?? ???????????."""
    center_lat = centers['center_lat'][country]
    center_lng = centers['center_lng'][country]

    return geopy.distance.geodesic([center_lng, center_lat], [lng, lat]).km


"""
????? ????????? ?????????? ????? ??????? ????, ??? ?????? lambda.
?? geopy.distance.geodesic ??????? ???????? ?????????.
????? ?? ???????????? 500 ???. ????? - ????????? ??? ?????????????? 
??? ??????? ????? ? ?????????? ?????????? ??????????.

"""

hotels_unique = hotels[['hotel_name', 'hotel_country', 'lng', 'lat']].drop_duplicates()
hotels_unique['hotel_distance_to_center'] = hotels_unique.apply(
    lambda row: get_dist_to_center(row['hotel_country'], row['lng'], row['lat']), axis=1)

hotels_unique.drop(['hotel_country', 'lng', 'lat'], axis=1, inplace=True)
hotels = hotels.merge(hotels_unique, how='left', on='hotel_name', sort=False)

# ?????? ??????? ? ???????????? ????? ???????:
hotels.drop(['lng', 'lat'], axis=1, inplace=True)

print(colored('\n???????? ????? ???????:', 'green'))
hotels[['hotel_name', 'hotel_distance_to_center']][:15]


### <span style="color:blue">??????? ???? ????? ('average_score')</span>

???????, ??? ??? ???? ?? ????? ??????????????? ?????????. ??? ?????????????? ???????? ?????, ? ??? ?? ??????????? ? ???????, ??? ? ??????? ??????????. ??????? ?????????????? ? ??? ????????? ?? ?????, ? ??? ??????? ?? ?????????? ??????? ?????????? ?????? ? ????????? ????? - ??????? ?????????.

#### <span style="color:blue">????? ?????????? ??????? ?? ????? ('total_number_of_reviews')</span>
#### <span style="color:blue">????? ?????????? ?????? ??? ?????? ('additional_number_of_scoring')</span>

????? ???-?? ??????? - ?????????? ??????? ? ???????????? ?/??? ??????? ?????. ??, ?? ????? ????????????? ????????, ?? ?????? ?????? ?? ?????? ???????. ??, ?????? ?? ?????? ???????????, ?????? ??????? ? ?????????????????? ???????? ????? ?????. ?????, ??? ???????? ?? ???????, ??????? ?????? ?? ?? ???????? ?? ????? ?????? ?????????.

*P.S. ??? ? ????????. ????? ?????????? ???????? ??????, ??????? ?????? ?????????? ??????? ????? ? ??? ??????. ?????????????, ?????, ??????? ????? ???? ??????? - ?????? ??? ????? ????? ? ???????? ????????, ??????? ????? ????????? ? ????? ???????? ?? ????? ? ????????? ? ??????????? ?????? ????????. ? ???????, ? ????? ?????? ?????? ???????? ????????? ???????.*

## 1.2 ?????? ????????? ?????????????? ?????????? ? ??? ???? ?????????? ? ?????

* ?????? ?????????? ('reviewer_nationality')
* ????, ??????? ????????? ??? ????? ('tags')
* ?????????? ???????, ??????? ?????????? ???? ? ??????? ('total_number_of_reviews_reviewer_has_given')

### <span style="color:blue">?????? ?????????? ('reviewer_nationality')</span>

    ?? ????? ????, ? ?????, ??? ?? ??????? ??????, ??????? ??????????????? ??????????? ?????????????? ???????????? ????????, ????????????? ????? ??????????? ?? ??????? ??????.
    
    ????????, ??????? ?? ?????-????????? ?????, ????? ??????? ?? ???????????? ? ???????? ???????? ? ??????????? ??????. ? ??? ???????????? ?????? ??? ?????????, ????????? ?????????????? ?????????? ? ??????? ??????????? ???????. ?????? ???????, ??? ?????? ????? ????? ??????? ????? ?????? ???????? ??????????? ? ?????.
    
    ? ??? ??????? ?? ??????????????? ????? - ????? ???? ?????????? ???????????? ??????? ??????, ??? ?????????, ??? ??? ?? ????? ????? ??????? ?????????? ?? ????????????.
    
    ??????? ?????????????? ????????:

In [ ]:
hotels['reviewer_nationality'] = hotels['reviewer_nationality'].apply(lambda x: x.strip())
nationality_count = len(hotels["reviewer_nationality"].value_counts())
print(f'????????????? {nationality_count} ????? ???????? ????????????.')

# ??????????????, ????????, 40 ????? ???????????????? ?????, 
# ?? ??????? ????????? ??????? ? ??????? ????? ??????:
print(colored('\n????? ?????????? ??????:', 'green'))
print(list(hotels['reviewer_nationality'].value_counts(ascending=False)[:40].index))

# ??????? ???????? ???????????? ??????:
english_list = ['United Kingdom', 
                'United States of America', 
                'Australia', 
                'Canada', 
                'New Zealand']
hotels['reviewer_en'] = hotels['reviewer_nationality'].apply(
    lambda x: 1 if x in english_list else 0)

# ??????? ????????????? ?????? ???????????????? ???????:
muslim_list = ['United Arab Emirates', 
               'Saudi Arabia', 
               'Turkey', 
               'Kuwait', 
               'Qatar', 
               'Oman']
hotels['reviewer_n_est'] = hotels['reviewer_nationality'].apply(
    lambda x: 1 if x in muslim_list else 0)

# ??????? ??????? ???????????? ??????? ?? ????????? ? ??????, ? ??????? ????????? ?????:
hotels['reviewer_resident'] = hotels.apply(
    lambda row: 1 if row['reviewer_nationality'] == row['hotel_country'] else 0, axis=1)

# ???????? ?????????? ? ?????? ??????????:
hotels['reviewer_nationality'] = hotels['reviewer_nationality'].astype('category')
hotels['reviewer_nationality_cat'] = hotels['reviewer_nationality'].cat.codes

#### <span style="color:green">????????????? ??????????? ??????? ???????? ??????? ?????????? ? ??????? ?????????? ?????????:</span>

In [ ]:
# ?????????? ?????? ??? ?????????:
mask = hotels['sample'] == 1
pivot_en = hotels[mask].groupby('reviewer_en')['reviewer_score'].mean()
pivot_n_est = hotels[mask].groupby('reviewer_n_est')['reviewer_score'].mean()
pivot_resident = hotels[mask].groupby('reviewer_resident')['reviewer_score'].mean()

pivot = pd.DataFrame({
    '?? ???????????? ??????': pivot_en,
    '?? ??????????????? ????????????? ??????': pivot_n_est,
    '???????????? ?? ????? ??????': pivot_resident 
}).T

# ?????? ?????????:
fig2, ax2 = plt.subplots(figsize=(16, 6))
pivot_barplot = pivot.plot(
    ax=ax2, 
    kind='bar', 
    color=['mediumslateblue', 'seagreen'], 
    rot=0)
ax2.set_title(
    '??????????? ??????? ??????????? ?????? ???????? ? ??????? ????? ?????????', size=16)
ax2.legend(['???', '??'], loc='upper right', fontsize=14)

# ??? ??????????? ??????? ?? ????????? ??????? ??????:
for p in pivot_barplot.patches:
    pivot_barplot.annotate('{:.2f}'.format(p.get_height()), (p.get_x()+0.1, p.get_height()),
    ha='center', va='bottom', fontsize=18)

plt.ylim(7, 9.5);

### <span style="color:blue">????('tags')</span>

In [ ]:
# ???????? ? ??????? ?????? ???????????? ????, ?? ??? ??????????:
def show_wordcloud(data, background_color, colormap):
    """?????? ?????? ???? ? ????????? ???????????."""
    wordcloud = WordCloud(
        background_color = background_color,
        colormap = colormap,
        max_font_size = 40,
        max_words=100,
        scale = 3,
        random_state = 42
    ).generate(str(data))

    plt.figure(1, figsize = (20, 20))
    plt.axis('off')
    plt.imshow(wordcloud)
    plt.show()

    
print('????????????? ????? ?????????? ???? ?????? ??????:')

# ????? ??????? ?? ???????????? ?????? ??????? ????, ? ?? ????????? ?????, 
# ??????????? ?????? ?????? ? ????? ??? ??????? ????:
def get_tags_string(tags_data):
    """????????? ??? ????? ?????? ???? ?????? ??????????????."""
    tags_data = tags_data.replace('[', '').replace(']', '').replace("'", '')
    tags_list = tags_data.split(',')

    tags_string = ''
    for teg in tags_list:
        teg = teg.strip().replace(' ', '_')
        tags_string = tags_string + teg + ' '

    return tags_string

# ?????? ?????????:
cloud = hotels['tags'].apply(get_tags_string)
show_wordcloud(cloud, 'white', 'viridis')

?????????? ? ?????????????? ???? ?????:

In [ ]:
hotels['len_tags'] = hotels['tags'].apply(lambda x: len(x.split(',')))

print(f'???????????? ?????????? ????? ? 1 ?????? ??????: {hotels["len_tags"].max()}/n')

print('???????:')
print(hotels[hotels['len_tags'] == hotels['len_tags'].max()]['tags'].iloc[0])
print(hotels[hotels['len_tags'] == hotels['len_tags'].max()]['tags'].iloc[1])

**????? ???????? 6 ????? ?????:**
* ??????????? ? ????????? ????????? 
* ??????????? ?? ?????? / ?????
* ??????????? ? ????? ??????? (????, ????, ???????? ??????, ????? ? ?????????? / ????????? ??????, ??????)
* ????????? ??????
* ???????????? ???????
* ?????? ?????????????? ? ?????????? ??????????



In [ ]:
"""
??????? ????????? ???????? ??????? ????????? - ?? ?????, ??? ???????????? 
?????????? ????? 6 ? ????? ?? ????????? ????. ??? ???, ????? ???? ???????, 
???????????? ? ??????????? ??????. ????? ?????? ? ?????? ?????? ???? ?? ????
????????? ???????? ???? ? ??????? ?? ?????? ????? ?????????. ???? ????????? 
?????-?? ??? ??? - ?? ? ????? ????? ??????.
"""

# ??????? ????? ????????:
def get_tags(tags_string):
    """????????? 6 ????? ????? ?? ????????."""
    tags_string = tags_string.replace('[', '').replace(']', '')
    tags_string = tags_string.replace("' ", '').replace(" '", '')
    tags_list = tags_string.split(',')
    tags_list = list(map(lambda x: x.strip(), tags_list))

    # ????? ???????? ??? ???????????????:
    reviewer_types = {'Solo traveler': 1,
                      'Couple': 2,
                      'Travelers with friends': 3,
                      'Family with young children': 4,
                      'Family with older children': 5,
                      'Group': 6}

    # ????????? ????? ?????????:
    tag_from_mobile = 0
    tag_with_pet = 0
    tag_leisure_trip = 1 #  ?? ????????? ?? ??????? ??????? ?????????????
    tag_reviewer_type = 0
    tag_count_nights = 0
    tag_room_type = ''
    
    # ???????????? ?? ????, ??????? ???? ? ???????? ??????:
    for teg in list(tags_list):
        if teg == 'Submitted from a mobile device':
            tag_from_mobile = 1
            tags_list.remove(teg)

        if teg == 'With a pet':
            tag_with_pet = 1
            tags_list.remove(teg)

        if teg == 'Business trip':
            tag_leisure_trip = 0
            tags_list.remove(teg)

        if teg == 'Leisure trip':
            tags_list.remove(teg)

        if teg in reviewer_types.keys():
            tag_reviewer_type = reviewer_types[teg]
            tags_list.remove(teg)

        if re.fullmatch(r'Stayed\s\d+\snight\w?', teg):
            tag_count_nights = [float(s) for s in re.findall(r'-?\d+\.?\d*', teg)][0]           
            tags_list.remove(teg)
            
        if len(tags_list) > 0:
            tag_room_type = tags_list[0]

    return tag_from_mobile, tag_with_pet, tag_leisure_trip, \
        tag_reviewer_type, tag_count_nights, tag_room_type


# ????????? ????? ????????:
hotels[['tag_from_mobile',
        'tag_with_pet',
        'tag_leisure_trip',
        'tag_reviewer_type',
        'tag_count_nights',
        'tag_room_type']
] = pd.DataFrame(hotels['tags'].apply(get_tags).to_list(), index=hotels.index)

print(colored('\n?????????? ?????????? ???????? ? ????? ?????????:', 'green'))
hotels[['tag_from_mobile', 'tag_with_pet', 'tag_leisure_trip', 
        'tag_reviewer_type', 'tag_count_nights', 'tag_room_type']].nunique()

?????????? ??????????? ????????? ???????. ???????? ????????? ???????? ?? ? ??????. ???????, ??? ?????? ? ?????????? ????????, ??????? ??????????????? ? ??????? ??????? ?????? ? ????? ? ? ????????? ??????? ????? ?????? ??????????.

    ?? ???? ?????? ????????????? ??????? ??????? - ????? ? ????? (?????, ??? ??????? ???? ??????? ?????? ????? ??????? ?????? ???? ???? ???????).

In [ ]:
# ???????????? ??? ?????? (??? ??????????? ?????????):
hotels['tag_count_nights'] = hotels['tag_count_nights'].astype('int8')

# ??????? ??????? ?????? ? ?????:
hotels['tag_view_room'] = hotels['tag_room_type'].apply(
    lambda x: 1 if ('View' in x) | ('Panoramic' in x) else 0)


# ?????????? ?????? ???????? ????????? ???????? ??????? 
# ? ???????? ??????????????? ???????? ?????? ??????:
def get_room_type(tag):
    """????????????? ???? ??????."""
    room_types = ['without Window', 'Guestroom', 'Classic', 'Single', 
                  'Standard', 'Superior', 'Comfort', 'Club', 'Suite', 
                  'Deluxe', 'King', 'Premier']

    for rt in room_types:
        if rt in tag:
            return rt

    if 'Double' in tag or 'Twin' in tag:
        return 'Double'

    return 'Other'


# ?????????? ?????? ? ????? ??????:
hotels['room_type'] = hotels['tag_room_type'].apply(get_room_type)

# ???????? ????? ????????
encoder_t = OneHotEncoder(cols=['tag_reviewer_type'], use_cat_names=True)
tag_type_bin = encoder_t.fit_transform(hotels['tag_reviewer_type'])
hotels = pd.concat([hotels, tag_type_bin], axis=1)

encoder_r = OneHotEncoder(cols=['room_type'], use_cat_names=True)
tag_rooms_bin = encoder_r.fit_transform(hotels['room_type'])
hotels = pd.concat([hotels, tag_rooms_bin], axis=1)

#### <span style="color:green">????????????? ??????? ????????????? ??????? ?????????? ? ??????? ?????????? ??????????? ? ????? ????, ? ????? ??????????? ?? ??????? ???????? ? ??????????? ?? ????????? ??????:</span>

In [ ]:
pivot_nights = pd.pivot_table(
    hotels[hotels['sample'] == 1], 
    index='tag_count_nights', 
    values='reviewer_score', 
    aggfunc=['count', 'mean'])
fig3, ax3 = plt.subplots(figsize=(15, 7))
plt.suptitle(
    '????????????? ?????????? ??????? ? ??????????? ?? ???????????? ??????????', size=20)
sns.barplot(x=pivot_nights.index, y=pivot_nights[('count', 'reviewer_score')]);

***?????????? ?????? - ??? ???????, ??????????? ???????? ??????? ????????????? 1-3 ????. ???????, ??? ????????? ??????. ? ???????????? ???????????? ?????????? - ?????????? ??????? ?????????? ?????????, ? ????? ?????????? ? 14 ????? (??????????? ????? ???????) - ?? ?????????? ?????? ?????.***

???????, ??? ? ????????? ?????? ??????????? ?????????? ? ???????????? ?????????? ? ?????. ?????? ???? ????????? ????????? ?????? ?? ??????????.
???????? ??? ???????, ?????? ?????? ?????????????? ?????? ? ????????? ????? ???????, ????????, ? ????? ? ?????????? ??? ??? ???????????.
???? ???????? ???????? ?? ?????? - ?? ??? ? ???????? ?????????? ??????. ?? ????? ???????? ??? ???????? ???????? ????????????.

***? ?????? ?????????, ???? ?? ??????????? ??????? ?????? ?? ????????? ??????:***

In [ ]:
# ????????????? ???????? ? ??????? ????????, ? ??????????? ?? ???? ??????:
pivot_types = pd.pivot_table(hotels[hotels['sample'] == 1], 
                             index='room_type', 
                             values='reviewer_score', 
                             aggfunc=['count', 'mean'])

fig4, ax4 = plt.subplots(figsize=(15, 7))
plt.suptitle('??????? ?????? ?? ??????? ? ??????????? ?? ????????? ??????', size=20)
bar_types = sns.barplot(
    x=pivot_types.index, 
    y=pivot_types[('mean', 'reviewer_score')])

for p in bar_types.patches:
    bar_types.annotate('{:.2f}'.format(p.get_height()), (p.get_x()+0.3, p.get_height()),
    ha='center', va='bottom', fontsize=20)

plt.ylim(7, 9.3);

??, ??? ???? ?????????????, ??????? ?? ?????? :D

In [ ]:
# ?????? ??????????????? ???????:
hotels.drop(['tag_reviewer_type', 'room_type', 'tag_room_type', 'len_tags'], 
            axis=1, inplace=True)

### <span style="color:blue">????? ?????????? ??????? ?????????? ('total_number_of_reviews_reviewer_has_given')</span>

???????, ??? ????? ????????, ??? ?????????? ??????? ? ?????????? - ?????? ?? ?????? ?? ?????? ?????. ?? ????????, ???????, ??? "?????????? ??????" ??????????????? ????? ????????????? ? ?????? ? ??????? ??????????. ??? ????????, ????, ??????? ????? ???????????? - ????? ????? ???? ????? ? ????? ?????? ????? ? ?????? ??????? ??????????. ? ??????, ???? ????????????????... ??? ???????? ????????? ? ????????. ??????? ??????? ??????? ?? ???????? ?? ???? ??????? ? ?????? ?????????.

In [ ]:
print(colored(
    '\n????????? ?? ?????????? ??????? ??????????? ?? ??????? ??????????:', 'green'))
print(hotels['total_number_of_reviews_reviewer_has_given'].groupby(
    pd.cut(hotels['total_number_of_reviews_reviewer_has_given'], 
           np.arange(0, 360, 25))).agg('count'))

# ????????????? ???????? ????????, ????? ???????? ?????? ?????????? 
# ? ??????? (???????) ?????????? ??????????:
hotels['total_number_of_reviews_reviewer_has_given'] = \
    hotels['total_number_of_reviews_reviewer_has_given'].apply(lambda x: np.log(x+1))

## 1.3 ????????????? ??????? - ???? ? ???????
* ???? ?????????? ?????? ('review_date')
* ??????? ????? ????? ?????? ? ????? ???????? ?????? ('days_since_review')

### <span style="color:blue">???? ?????????? ?????? ('review_date')</span>

? ???? ???? ????????, ??? ?? ?????? ???????, ???????? ?????????? ????????????, ????? ?????? ?? ?????? ???? ?????????? ? ?????, ?? ? ????????? ??????????? ? ?????.

- ?? ??????, ??? ?????? ???????? ??? ??????????? ? ??????? ??????????? ??????, ?????? ??????? ?????? ?????? ?????????? ? ?????????????? ??????????. ????? ??????????? ??????????? ???????????? ? ??????? ?????????????? ???????, ? ? ?????? ????? ?????????????? ??????????? ?????????. ???????, ??? ?????? ?????? ?????? ?? ????? ?????? ?????.
- ?????, ????? ?????????? ????? ??? ????????? ??????????? ???????. ??? ???????? ????? ??? ????????, ?? ??? ???? ?????????? ????????.
- ????, ???????, ??????? ??????. ??? ???? ???????? ? ???? ????????? ????? ????? ???????????. ?? ???????????? ????????? ?????? ??? ?????? ????? ?????????? ????, ????????, ???????? ? ?????????????, ??? ??????? ?? ????????. ???????? ????? ?? ??????????? ? ??????? ??????? ???????????, ??????, ??????????? ??? ?????? ????????? ????? ????? ????? ?????? ?? ????? ??????.
- ?? ? ?????. ?????? ? ??????????? ?????? ???? ?? ?????? :D. ?????????? ???????? ?????????? ???????, ? ??????? ????? ??????? ? ????. ????? ??????? ?????????????? ?????? ?????? ?????? ?????? ????? ?????????? ????????? ?????? ? ?????????????? ?????????, ??? ????? ??????? ?????????????? ?????????? ???????????.

??? ???????, ??? ???????? ????? ???????? ? ????????? ???????? - ?????, ?????? ? ????? ???? ????????? ?????.

*PS. ? ??????????? ? ?????????? ?? ???? ???????????? ????? ???? ? ????????? ???????. ? ?????????? ? ?????????????? ??????????? ?????? ???? ???????? ???????? MAPE.*

In [ ]:
# ??????????? ?????? ? ??????? ? ?????? ????/???????:
hotels['review_date'] = pd.to_datetime(hotels['review_date'])

# ????????? ????? ????????:
hotels['review_week'] = hotels['review_date'].apply(
    lambda x: x.isocalendar()[1])
hotels['review_month'] = hotels['review_date'].dt.month

#### <span style="color:green">????????????? ??????? ?????? ????????? ????? ?? ??????? ?????????? ????????:</span>

In [ ]:
mask = hotels['sample'] == 1
pivot_month = hotels[mask][['review_month', 'reviewer_score']].groupby(
    'review_month').mean()
fig5, ax5 = plt.subplots(figsize=(15, 5))
plt.suptitle('??????? ?????? ?? ??????? ? ??????????? ?? ??????? ????', size=16)
bar_month = sns.barplot(
    x=pivot_month.index, 
    y=pivot_month['reviewer_score'])
ax5.set_xticklabels([
    '???', '???', '???', '???', 'MA?', '???', 
    '???', '???', '???', '???', '???', '???'])

for p in bar_month.patches:
    bar_month.annotate(
        '{:.2f}'.format(p.get_height()), (p.get_x()+0.4, p.get_height()),
    ha='center', va='bottom', fontsize=14)

plt.ylim(8, 8.7);

### <span style="color:blue">??????? ?????? ? ???? ??? ???????? ?????? ('days_since_review')</span>

??? ???? ???????????? ?? ???? - ??? ????? ??????? ?????? ?????? ?? ??????. ????? ????????? ????????, ????? ???, ??? “?????? ???? ???? ??????”. ?? ??? ??? ?????? ???????? ? ????? ?????? ?????????????? ????? ???? 2 ??????, ???? ???, ?? ?????, ?? ???????????? :D
?????????? ? ????????? ?????????? ??? ???????? ?? ????, ? ???? ??????? ??????, ?? ????????? ????????, ?? ?????????? ?? ????? ?????? ?????????.

In [ ]:
hotels.drop('days_since_review', axis = 1, inplace=True)

## 1.4 ??????
- ????????????? ?????, ??????? ????????? ??? ????? ('negative_review')
- ????????????? ?????, ??????? ????????? ??? ????? ('positive_review')
- ????? ?????????? ???? ? ????????????? ?????? ('review_total_negative_word_counts')
- ????? ?????????? ???? ? ????????????? ?????? ('review_total_positive_word_counts')

????????? ?????? ?? ????? ?? ????? “Booking.com” ???????? ????? ???????, ??? ???????? ? ???? ??? ?????: ????????????? ????? ? ????????????? ?????. 

????? ??????????? ?????? ?? ?????????? ?????????? ???????????? ???????? ?? ??? ???????:
- ??? ??? ?? ??????????? ? ??????? ???????????
- ??? ??? ??????????? ? ??????? ???????????

????? ??????? ?????????? ??? ????? ?????? ?? ???? ??????.

### <span style="color:blue">?????????? ???? ? ??????</span>

???? ????????? ??????????? ?????????? ???? ? ?????? ? ??????? - ?? ??? ????? ?????????? ?? ?????? ?? 1-2 ??????.
? ?????????? ???? ?????? 0 - ????????????? ?????? 'No Positive' / 'No Negative', ?? ???? ???????????? ?????????? ??????.
?? ?????? ???? ???? ????????? ? ???????? ?????? ? ???? ??????? ??? ??????????????.

***??? ???????? ??? ???????? ???? ??? ?????????? ? ???, ???????????? ?? ?????? ??????????? ? ?????. ? ??????????? ???????, ???? ??? ???? ?????????, ?? ??????: ??? ??? ?? ???????????? - ?? ?? ???????? ?????? ??? ?????????? ??????????? ?????? ???? "??? ???????????" ??? "??? ???? ?????????!". ???? ?? ???? ?? ??? ???????????? - ??? ????? ?????? ????? ??????????? ???????. ? ????????????? ?????? - ?? ?? ?????. ???????? ??? ????????:***

In [ ]:
fig6, ax6 = plt.subplots(figsize=(15, 7))
plt.suptitle('??????????? ?????????? ???? ? ???????????? ??????', size=16)
line_count_neg = sns.lineplot(
    data=hotels[mask],
    x='reviewer_score',
    y='review_total_negative_word_counts',
    color='red', linewidth=2
)
line_count_pos = sns.lineplot(
    data=hotels[mask],
    x='reviewer_score',
    y='review_total_positive_word_counts',
    color='green', linewidth=2
)
ax6.set_xlabel('')
ax6.set_ylabel('')
ax6.legend(['? ?????????? ??????', '', '? ?????????? ??????', ''], 
           loc='upper right', fontsize=12);

### <span style="color:red">?????????? ??????????? ??? ??????? ??????? ?? Booking.com</span>

?? ????? ????, ????? ??????? ???????, ??? ????? ??????????. ??? ??????? ?? ?????????? ??????????? ?????? ?? ????? Booking.com.
?? 2019, ??????? ????, ??????? ?????? ??? ???? ?????????:

In [ ]:
Image(filename='../input/projectdata/img_score.png', width=600)

????????? ????????? 6 ????????? ? 4 ????????. ?????? ????????? - 10 ?????? ????????, ?? ?????? ??? + 2,5 ????? ? ????? ?? ?????????, ? ???????? ?????? - ??????? ?????????????? ? ??????????? ?? ???????.

???? ??????????????? ????????????? ???? ?????? - ?? ?????? ???????? ???????? ??? ??? ?? ???? 19 ?????????:

In [ ]:
fig1, ax1 = plt.subplots(figsize=(15, 7))
plt.suptitle('????????????? ?????? ????????? ?? ????????', size=20)
sns.histplot(
    ax=ax1, 
    data=hotels[mask]['reviewer_score'], 
    bins=len(hotels['reviewer_score'].value_counts())
).xaxis.grid(linestyle='--')
ax1.set_xticks(np.arange(0, 10.5, 0.5))
ax1.set_xlabel('');

In [ ]:
# ???????? ?????? ???? ?????? ? ?? ???????????? ?? ??? ?????
# 1) ?? ??????? ???????????? ? ??????? ??????
# 2) ?????????
scores_list_1 = hotels[mask]['reviewer_score'].value_counts().iloc[:19]
scores_list_2 = hotels[mask]['reviewer_score'].value_counts().iloc[19:]

print(colored('\n??????, ?????????????? ? ??????? ?????????? Booking.com:', 'green'))
display(scores_list_1)

print(colored('\n??????, ?? ?????????????? ? ??????? ?????????? Booking.com:', 'green'))
display(scores_list_2)

 ???? ?????????? ?? ???????? ????? ?????????? ?????? - ?? ??? ?? ????? 10, 9.6, 9.2 ? ??? ?????. ?????? ???? ?????????? ????????????? (??? ???????????????) ?????????????.

? ??? ?????? ??????? ????????? ??????, ? ????? ?? 0.5 - ?? ????? ???????. ????????? ?? ?????????? - ?????? ???????? ?? ?????? ?????. ????? ???, ????????, ?????????? ???????????? ?????????????? ??????? ??????????, ??? ?????? ???????????? ??????, ??? ??? ???-??.

In [ ]:
# ? ?????? ????????????? ????? ?????????? ???? ? ?????? ? ?????? ?????? 
# ??? ????? ???? 1% ?????????? ????????:
hotels_filtered = hotels[
    mask & (hotels['reviewer_score'].isin(scores_list_1.index.to_list()))]

fig7, ax7 = plt.subplots(figsize=(15, 7))
plt.suptitle(
    '??????????? ?????????? ???? ? ??????? ? ???????????? ??????\n (?????????? ????????)', 
    size=16)
line_count_neg = sns.lineplot(
    data=hotels_filtered,
    x='reviewer_score',
    y='review_total_negative_word_counts',
    color='red', linewidth=2
)
line_count_pos = sns.lineplot(
    data=hotels_filtered,
    x='reviewer_score',
    y='review_total_positive_word_counts',
    color='green', linewidth=2
)
ax7.set_xlabel('')
ax7.set_ylabel('')
ax7.legend(['? ?????????? ??????', '', '? ?????????? ??????', ''], 
           loc='upper right', 
           fontsize=12);

### <span style="color:red">????? ??????????? ???????????.</span>

#### ????????? ?? ????????????? ???? ?????????:

In [ ]:
# ????????? ??????? ????????:
fig10, ax10 = plt.subplots(nrows=2, ncols=1, sharex='col', figsize=(15, 8))
plt.suptitle('????????????? ?????????? ???? ? ???????', size=20)

sns.boxplot(
    ax=ax10[0], 
    data=hotels['review_total_positive_word_counts'], 
    color='green', 
    width=0.4, 
    orient='h'
).grid(linestyle='--')

sns.boxplot(
    ax=ax10[1], 
    data=hotels['review_total_negative_word_counts'], 
    color='red', 
    width=0.4, 
    orient='h'
).grid(linestyle='--')
ax10[0].set_title('? ????????????? ?????:', ha='left', x=.0, fontsize=14)
ax10[1].set_title('? ????????????? ?????:', ha='left', x=.0, fontsize=14);

????????????? ????????????? (??????? ?? ????????????? ???????? ????????), ??????? ??????? ???????? ?? ????????? ??????? ???????? ???????? - ??? ??????? ?? ????????, ? ????????? ???? ??????. ????????????????? ??????? ?????????? ???? ? ???? ?????? ?????? - ??? ???? ?? ????? ??????? ???????????? ? ???? ????????.

??????? ? ???? ???? ???? ???-?? ?????????? ?????????? ????, ?????????? ?? ?????? ????? ???, ????????, ??????????????? ??????. ?? ?? ???? ?? ???? ?????????????? ?? ?????? ????????? ????????? ???????? ??????, ? ? ?? ??? ??????????. ???? ?? ?????, ??????? ??????? ???? ?????? ? ??????????? - ?? ?????? ?????? ?? ???????????? ???????? ?????????? ?????? (Random Forest).

????????? ?????? ? ?????? ??????????? ???? ????? ??????, ???? ????? ??????? ????????? ?? ????? ?????.

??????? ?????????? ????????????? ??????????? ?? ????? ???????. ???????? ??? ??????? ? ????, ??????? ??????? ?????????? ???? ? ?????????? ? ?????????? ?????? ?????? (?????????????? ??????????? ??? ?????):

In [ ]:
mm_scaler = MinMaxScaler()

hotels['review_total_positive_word_counts_norm']  = pd.DataFrame(
    mm_scaler.fit_transform(pd.DataFrame(hotels['review_total_positive_word_counts'])),
    columns=['review_total_positive_word_counts'])

hotels['review_total_negative_word_counts_norm']  = pd.DataFrame(
    mm_scaler.fit_transform(pd.DataFrame(hotels['review_total_negative_word_counts'])),
    columns=['review_total_negative_word_counts'])

# ???????? ??? ?????????? ? ????, ????????????? ????? - ? ????????????? ?????????????:
hotels['review_total_word_counts_diff_norm'] = \
    hotels['review_total_positive_word_counts_norm'] - \
    hotels['review_total_negative_word_counts_norm']

# ? ????? ??????? ???? - ??????????? ?? ?????????? ????? ??? ??????????? 
# ? ???????????? ??????????????? ??????????:
hotels['positive_words_more'] = hotels['review_total_word_counts_diff_norm'].apply(
    lambda x: 1 if x >= 0 else 0)

hotels.drop(
    ['review_total_positive_word_counts_norm', 'review_total_negative_word_counts_norm'], 
    axis=1, inplace=True)

????? ??????? ???? ????? ?? ?????? (?????? ?????????? ????? ??????? ????????????????, ???????? - ?? ????????):

In [ ]:
hotels['review_total_word_counts'] = \
    hotels['review_total_positive_word_counts'] + \
    hotels['review_total_negative_word_counts']

# ???? ????? ?????????? ???? ????? 0, ?? ??????? ??? ????? ????????????? (?? 0.5):
hotels['review_positive_word_proportion'] = hotels.apply(
    lambda row: 0.5 if row['review_total_word_counts'] == 0 \
            else row['review_total_positive_word_counts'] / row['review_total_word_counts'],
axis=1)

# ??????? ??????????????? ???????:
hotels.drop('review_total_word_counts', axis = 1, inplace=True)

### <span style="color:blue">??????????????? ????? (‘negative_review’, ‘positive_review’)</span>

? ?????? - ????? ??????????:D 
???????? ?????????? ?????????? ??? ??????? ???????????? ???????? ?????? ?????????? ?????? ? ???? ????????! 

?? ??? ???? ????? ?????????????? ??????? ??? ????? - ????? ????? ??????????????? ?????? ??????. ? ???????? ??? ???????:
*     VADER (Valence Aware Dictionary and sEntiment Reasoner) — ??? ?????????? ?? ??????? ? ???????? ?????????? ??????? ??????????, ?????????? ??????????? ?? ???????, ?????????? ? ?????????? ?????.

    [?????????? ?? GitHub ???](https://github.com/cjhutto/vaderSentiment)

* TextBlob — ??? ?????????? Python (2 ? 3) ??? ????????? ????????? ??????. ?? ????????????? ??????? API ??? ?????????? ? ????? ?????? ????????? ????????????? ????? (NLP), ????? ??? ??????????? ?????? ????, ?????????? ??????? ?????, ?????? ???????????, ?????????????, ??????? ? ?????? ??????. 

    [?????? ?? ???? ?????????? ??? ](https://textblob.readthedocs.io/en/dev/)

* ??????????? ???? ????????????? ??????

?????? ??? ???????????????? ???????????? ? ???????? ???????????? ????? ?????????? ???? (??? ????? ????????? 30 ??????? ????):

In [ ]:
# ????????????? ???????? ????? ????????????? ? ????????????? ?????? ???????
# ????????? ??????? ??? ????????????
nlp = spacy.load('en_core_web_sm')
stop = stopwords.words('english')
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()


def clean_text(text):
    """??????? ?? ????-???? ? ???????????? ??????"""
    text = text.lower()
    text = [word.strip(string.punctuation) for word in text.split(" ")]
    text = [word for word in text if not any(c.isdigit() for c in word)]
    text = [word for word in text if word not in stop]
    text = " ".join(text)
    return [lemmatizer.lemmatize(w) for w in w_tokenizer.tokenize(text)]


def get_words_string(words_data):
    """?????????? ????????? ???? ? ?????? ??????"""
    words_string = ''
    for word in words_data:
        words_string = words_string + word + ' '
    return words_string


def draw_cloud(text_column, background, color_map):
    """???????????? ????????? ???????"""
    hotels['temp'] = hotels[text_column].apply(
        lambda x: x.replace('No Negative', '').replace('No Positive', ''))
    hotels.temp = hotels[text_column].apply(clean_text)
    cloud_data = hotels.temp.apply(get_words_string)
    # ????????????? ???????? ??? ???????????? ?????:
    show_wordcloud(cloud_data, background, color_map)

In [ ]:
print('\n?????? ???? ?????????? ???????:\n')
draw_cloud('positive_review', 'white', 'Dark2');

In [ ]:
print('\n?????? ???? ?????????? ???????:\n')
draw_cloud('negative_review', 'white', 'BrBG');

### ?????? ?????????? ? ??????? SentimentIntensityAnalyzer ?????????? VADER:


  SentimentIntensityAnalyzer ?? VADER ??????? ?????? ?? 3 ??????? ?????????????, ? ????? ????????? ??????. ? ?????????? ?? ??????? ??????? ??????????????? ?????? ??????????? ?????? ?? ??????????:
- ?????????? ??????
- ??????????? ??????
- ??????????

  ?? ???????????? VADER ?? ????????? ??????: ????????? ?????? ??????????? ????? ???????????? ?????? ??????????? ??????? ????? ? ?????????, ?????????????? ? ???????????? ? ?????????, ? ????? ????????????? ???, ????? ???? ? ????????? ?? -1 (??????? ????????????? ????????) ?? +1 (??????? ????????????? ????????).

? ??? ?? ??????, ?? ???? ? ?? ?? ????? ? ????????????? ? ????????????? ?????????? ????? ????? ??????????????? ???????????.

### “???” ??? “??????”?

????????, ??? ????????????? ? ????????????? ????? ?????? - ??? ?????? ?? ??? ??????? ????? ? ??? ??? ??????????? / ?? ??????????? ? ?????. ? ????????? ??????????? ????? ?????? (??? ??????????), ??? ????? ?? ?????? “??? ??? ?? ????????????”: “??????” (nothing) - ????? ????????, ??? ?????? ?? ?? ???????????, ?? ???? - ??? ???????????. ????? ????????????? ????????. ?? ???? ????? “??????” ? ????? ?? ??????: “??? ??? ????????????” ????? ???????? ????? ????????.

? ??? ????? “???” (everything) ????? ????????? ?? ????? (????? ??).

??? ????? ?????????? ??? ??????????? ????? ??????????? ???????????, ?? ? ???????? ?????? ?? ??????, ?????? ???? ?? ??? ??????????? ?????? ????? ????? ?????????? ???? ????????. ?????? ?????? ??????? ??? ?????? ?????? ???????. 

???? ????????, ??? ???????, ??? ????? ????? ??????????? ?? ?????? ? ???????? ???????, ?? ? ? ??????????? ????????????, ??? ??? ?????? ?????? ????? ?? ?????. ?? ? ?? ????? ????? ???????? ???? ? ????? ??????. ?? ????? ????? ?????????? ??????????. 

??? ?? ????? “small” ?? ???????? ? ???????? ????? ????????????? ??????? - Small room, Small bed, Small pool. ? ??????? ??????????? small ? ????????????? ??????? ?? ?????? ????????? ?????????? ????????.

??? ?????? ? ??????? ??????? ??????? ? ?????? ??? ?????? ??????????? - SentimentIntensityAnalyzer, ? ???????? ??????? ??????? ??? ??????????????? ????? ???????. 

  ? ??? ???? ??????. ??????, ??????? ‘No Negative’ / ‘No Positive’ ???????????? ?????????? ??????. ???? ???????? ??? ??? ????, ?? ?????????? ???????? “No”, ?????? ??????????? ???????????????? ?? ???????? ?????. ????? ?????? ??????? ??? ????????, ??????? ?? ?????? ??????, ?? ??? ??? ?????????? ???????????? ?????? - ???????? - ?????????? ?????????????? ????? ?????????? ? ????????, ?? ????? ??????? ??? ????? ??? ???????, ?????????????? ???????????????? ????? ??? “??”. ? ??? ???? ?? ???????? ?????? ??????, ?? ???????? ????????????????? ??????????? ????? ????? ? ????? ????? ??????????? 0.

In [ ]:
# ??????? ??? ?????????? ???????????, ????? ????????? ???????? 
# ??? ?????? ??????? ?????? ????????????? ? ????????????? ???????:
sia_pos = SentimentIntensityAnalyzer()
sia_neg = SentimentIntensityAnalyzer()

# ????????? ??? ????????? ?????????????? ????? 'nothing':
example = 'nothing'
print(f'??????? ?????? ????? "{example}" ?? ??????????????:')
print(sia_neg.polarity_scores(example), '\n')

# ??????? ????????:
neg_words = {'small': -1, 'nothing': 10}
pos_words = {'nothing': -10, 'everything': 10}
sia_neg.lexicon.update(neg_words)
sia_pos.lexicon.update(pos_words)

print(f' - ????? ?????????????? (??? ??????????? ?????????):')
print(sia_neg.polarity_scores(example))
print(f' - ????? ?????????????? (??? ??????????? ?????????):')
print(sia_pos.polarity_scores(example))

# ????? "No Negative" ???????? ????? ????? ?????????? ???????:
example = 'No Negative'
print(f'\n??????? ?????? ????? "{example}":')
print(sia_neg.polarity_scores(example))
example = 'Positive'
print(f'??????? ?????? ????? "{example}":')
print(sia_neg.polarity_scores(example))

# ?????? ????? ?????????? ?????? ??????:
hotels['negative_review_temp'] = hotels['negative_review'].apply(
    lambda x: x.replace('No Negative', 'Positive'))
hotels['positive_review_temp'] = hotels['positive_review'].apply(
    lambda x: x.replace('No Positive', 'Negative'))

In [ ]:
%%time
# ??????????)
hotels['negative_review_sentiments'] = hotels['negative_review_temp'].apply(
    lambda x: sia_neg.polarity_scores(x))
hotels['positive_review_sentiments'] = hotels['positive_review_temp'].apply(
    lambda x: sia_pos.polarity_scores(x))

# ????????? ????? ????????:
hotels['n_review_sentiments_neg'] = hotels['negative_review_sentiments'].apply(
    lambda x: x['neg'])
hotels['n_review_sentiments_neu'] = hotels['negative_review_sentiments'].apply(
    lambda x: x['neu'])
hotels['n_review_sentiments_pos'] = hotels['negative_review_sentiments'].apply(
    lambda x: x['pos'])
hotels['n_review_sentiments_compound'] = hotels['negative_review_sentiments'].apply(
    lambda x: x['compound'])

hotels['p_review_sentiments_neg'] = hotels['positive_review_sentiments'].apply(
    lambda x: x['neg'])
hotels['p_review_sentiments_neu'] = hotels['positive_review_sentiments'].apply(
    lambda x: x['neu'])
hotels['p_review_sentiments_pos'] = hotels['positive_review_sentiments'].apply(
    lambda x: x['pos'])
hotels['p_review_sentiments_compound'] = hotels['positive_review_sentiments'].apply(
    lambda x: x['compound'])
# ??????, ?? ????? ????? ??????? (???? ??? ?????!)

### ?????? ?????????? ? ??????? TextBlob:

TextBlob ???????? ??????? ?? ???????. ? ??????? ?? Vader, ?? ??????????? ?? ?????, ? ????? ? ???????????, ??????? ????? ????????? ???: ?????????? ? ??????????????.

?????????? — ??? ????? ? ????????? ??????? ?? -1 ?? 1, ??? -1 — ????????????? ???????????, ? 1 — ????????????? ???????????.
?????????????? — ??? ?????????? TextBlob, ????????????, ????????? ?? ??????????? ? ??????? ??????? ?????? ??????? ??? ??? ???????? ?? ??????.

??? ?????? ?????????? ?????????? ????? ???????????? ???????? ??????????.

????? ??? ?????? NaiveBayesAnalyzer ????? ???????? ????????????? ?????????? (??????????? Vader) ? ?????????? ? ?????????? ?????? ?????? ?, ?? ?????? ???? ?????? ??????? ????????? ??????, ?????????????? ????????? ???????:

***- (????????????) ??? ????? ????? ?????????? ???????????: ?????? ?????????***
??????? ??? ??? ???????????, ??? ? ? ?????? ?? ?????? ? ??????????? - ?? ???????? ????????????? ??????

***- (??????????????) ?????????? ????? ????? ?????????? ???????????, ?????????? ????? - ??????????: ??????? ??????? ?????????***
??????????? ????? ?????????? ? ??????????? ?????

***- (???????????????) ?????????? ????? ????? ?????????? ???????????, ?????????? - ??????????: ??????? ?????? ?????????***
?????????? ?????? ?? ???????????, ?? ????? ? ??????????? ?? ???????

***- (??????????????) ??? ????? ????? ?????????? ???????????: ?????? ?????????***
?????? ?? ???????????

?????? ?? ????? ?????????? ????? - ??????? 4 ?????? ? ????????? ?? 2.5 ?? 10 ? ?????? ?????.

P.S. ??? ? ??? ?? ????, ??? ???? ???????-????????????? ?????? ????? ???????? ??????? ??-?? ?????? ?????????? ? ?????? ????????

In [ ]:
# ??? ??? ???? ?????????? ???????? ?? ?? ???????, ? ? ???????, 
# ?? ?? ??????????? ? ??????????? 'No Negative' / 'No Positive':
text_example = 'No Negative'
example = TextBlob(text_example)
print(f'?????????? ????? "{text_example}": \
{round(example.sentiment_assessments.polarity,2)}')

text_example = 'No Positive'
example = TextBlob(text_example)
print(f'?????????? ????? "{text_example}": \
{round(example.sentiment_assessments.polarity,2)}')

In [ ]:
%%time

def get_polarity(text):
    """????????? ????????? ?????????? ??????"""
    blob = TextBlob(text)
    return blob.sentiment_assessments.polarity

hotels['positive_polarity'] = hotels['positive_review_temp'].apply(get_polarity)
hotels['negative_polarity'] = hotels['negative_review_temp'].apply(get_polarity)

In [ ]:
Image(filename='../input/projectdata/wait.png', width=600)

??? ?? ? ?????????? ???? ?????? ???????? ???????????? ?????????? ? ?????????? ??????? (? ????? ???? 1) ? ???????????????? ????? ? ??????????? ?? ????, ????? ????????? ???????????.

In [ ]:
%%time

tb = Blobber(analyzer=NaiveBayesAnalyzer())

text_example = 'everything was great'
print(f'{tb(text_example).sentiment} ??? ????? "{text_example}"')

# ??? ??? ?????????? ? ?????????? ????????? - ??????????????? ????????, ?????????? ? ????? 1 - 
# ????? ??????????? ?????? ?????:
def get_blobber_sentiment(text):
    """????????? ???????????? ?????????? ??????"""
    blobber_sentiment = tb(text).sentiment
    return blobber_sentiment[1]

hotels['positive_review_pos'] = hotels['positive_review_temp'].apply(get_blobber_sentiment)
hotels['negative_review_pos'] = hotels['negative_review_temp'].apply(get_blobber_sentiment)

In [ ]:
# ??????? ?????? ????????????? ? ??????????? ?? ???? ?????????????? ??????????:
hotels['positive_review_class'] = hotels['positive_review_pos'].apply(
    lambda x: 1 if x >= 0.5 else 0)
hotels['negative_review_class'] = hotels['negative_review_pos'].apply(
    lambda x: 1 if x >= 0.5 else 0)


# ????????? ???????? ? ?????????????? ??????:
def get_class(positive_review_class, negative_review_class):
    """????????????? ??????, ? ??????????? ?? ??????? ????? ??????"""
    if positive_review_class + negative_review_class == 2:
        return 10.0
    if positive_review_class == 1 and negative_review_class == 0:
        return 7.5
    if positive_review_class == 0 and negative_review_class == 1:
        return 5.0
    return 2.5


hotels['review_classification'] = hotels.apply(
    lambda row: get_class(row['positive_review_class'], row['negative_review_class']), 
    axis=1)

### ??????????? ????????????:

??????????? ???????? ?? ???????? ????????? ?????????????, ?????????? ?? ??????? ??????????, ?? ?? ??????????? ????????????? ???????? ???????????, ??????? ?????? ?? ??????. ?? ????? ????, ???? ???? ?????????????? ???????, ??????????? ??????????? ????????????, ?? ??? ??????? ??????. ? ??????? ??????, ??? ????????? ?????????? ?? ????? ? ????? ????? ????????? ???????.

? ??????????? ????? ??????? ?????? ???????? ??? ?????? ? ??????? ? ??????? ?????-?? ??????????? ?????????. ?? ??? ??????, ????? - ??????? 

? ???????? ??????? ????? ????? ???? ???????? ??? ????????????? ???????, ??????????? ?? ????????????? ????? ??????? ? ?????? ?????? (? ????? ??????? ??????? ?????????). ????? ??????, ????? ??????? ???? ??????? ?????? ???????????? ? ???? ???????.

(?????? ? ?????????????? ???????? - ???? ?????? ?? ????????, ??? ??? ???? ????? ????????????? ?????? ????? ????? ????????????, ???? ???? ?? ?????? ??????? ??????.)

In [ ]:
%%time

# ????????? ?????? ???? ?????? ????? ?????? ?????? ? ??????????? ?? ????? ?????????? ????:
hotels['negative_review_temp'] = hotels['negative_review'].apply(
    lambda x: x.replace('No Negative', ''))
hotels['positive_review_temp'] = hotels['positive_review'].apply(
    lambda x: x.replace('No Positive', ''))
hotels['review'] = hotels['positive_review_temp'] + ' ' + hotels['negative_review_temp']


# ??????? ?????????????? ?????? ? ????? ????:
def get_words_set(text):
    """???????????? ??????? ?????????? ???? ??????"""
    text = text.lower()
    text = [word.strip(string.punctuation) for word in text.split(" ")]
    text = [word for word in text if not any(c.isdigit() for c in word)]
    text = [word for word in text if len(word) > 2]
    text = [word for word in text if word not in stop]
    text = " ".join(text)
    words_data = [lemmatizer.lemmatize(w) for w in w_tokenizer.tokenize(text)]
    words_set = set()
    for word in words_data:
        words_set.add(word)
    return words_set


# ????????? ?????? ?????????? ???? ??????? ??????:
hotels['review_word_set'] = hotels['review'].apply(get_words_set)

# ???? ??????!

In [ ]:
best_words_set = set()

# ???????? ??????? ?????? ??????? ? ?????? ??????:
hotels_top = pd.DataFrame(hotels[hotels['average_score'] >= 9.6])
hotels_top['positive_review_word_set'] = hotels_top['positive_review'].apply(get_words_set)
hotels_top_list = hotels_top['positive_review_word_set'].tolist()

for review_set in hotels_top_list:
    best_words_set.update(review_set)

best_hotels_count = len(hotels[hotels['average_score'] >= 9.6]['hotel_name'].value_counts())

print(f'?????????? ?????????? ????, ?????????? ?? ??????????? {len(hotels_top_list)} ??????? '
      f'{best_hotels_count} ?????? ??????: {len(best_words_set)}.')

In [ ]:
def get_diff_with_top(review_words_set):
    """??????????? ???? ???? ??????, ????????????? ????? ? ??????? ? ???-??????"""
    set_len = len(review_words_set)
    if set_len == 0:
        return 0
    top_words_count = 0
    for word in review_words_set:
        if word in best_words_set :
            top_words_count += 1

    return round(top_words_count / set_len, 2)


# ????????? ????? ???????:
hotels['review_difference_with_top'] = hotels['review_word_set'].apply(get_diff_with_top)

### ?????! ???????? ??? ? Afinn:

?? ????? ????, ??????? ????? ?????????-??????? - ???????? ?????????, ???? ????????? ? ?????? ????????????. ??, ??????? ???????????? ? - ??????? ????? ?????????? ? ???????? ???????? ???????.

??????, ??????? ?? ?????????? ????????? ?????????, ? ?????????? ??? ?? ???? ?????????? ? ??????? ?????? ???????? ??????????? ?????????? ??????. Afinn — ??? ?????? ? ??????? ?????????? ?? ?????? ?????? ????. ?? ?? ??????????? ?????? ? ?? ???????? ???????????, ??????????? ? ????????? ?????????? ???? ??????. ?? ????, ?????? ???? ?????????? ?? ?????????? ???? ??????:

In [ ]:
%%time
# ???????? ????:
afinn = Afinn()
hotels['afinn_score'] = hotels['review'].apply(lambda x: afinn.score(x))

#### <span style="color:green">????????????? ??????????? ?????? ????????? ? ????????????? afinn:</span>

In [ ]:
# ???????? ???????? ????? ??? ????? ???????:
hotels_filtered = hotels[mask & (hotels['reviewer_score'].isin(
    scores_list_1.index.to_list()))]
pivot = hotels_filtered.pivot_table(
    values='review', 
    index='reviewer_score', 
    columns='afinn_score', 
    aggfunc='count')

# ?????? ?????????:
fig8, ax8 = plt.subplots(1, 1, figsize = (15, 6))
afinn_heatmap = sns.heatmap(data=pivot, cmap='YlGnBu')
ax8.invert_yaxis();

In [ ]:
# ????????? ??????? ????????? ???????? ?? ?????? ?????:
def get_outliers_bonds(data, feature):
    """?????????? ??????? ?????????? ??? ??????????? ????????"""
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower = quartile_1 - (iqr * 1.5)
    upper = quartile_3 + (iqr * 1.5)
    return lower, upper


lower_bound, upper_bound = get_outliers_bonds(hotels, 'afinn_score')

hotels['afinn_score'] = hotels['afinn_score'].apply(
    lambda x: lower_bound if x < lower_bound else (upper_bound if x > upper_bound else x))

# 2. ?????? ? ????? ????????? (FEATURE SELECTION):

In [ ]:
# ??????? ???????? ???? ??? ?????????? ??????????? ?? ????????? (??? ??????????? ?????????????):
hotels_copy = hotels.copy()

In [ ]:
print(f'??????? ???????????? ?????????? ?????????: {len(hotels_copy.columns)}')
print('(??????? ??????? ??????? ? ????? ????/?????)')

# ?????? ?????????? ??????:
object_columns = [s for s in hotels.columns
                  if ((hotels[s].dtypes == 'object') | \
                      (hotels[s].dtype.name == 'category') | \
                      (hotels[s].dtypes == 'datetime64[ns]'))]

print(f'?????? {len(object_columns)} ???????? ? ??????????? ???????')
hotels_copy = hotels_copy.drop(object_columns, axis = 1)

### ???????? ???????? ???????? ?? ??????:

In [ ]:
# ??????????? ? ?????????????? ????????:
num_cols = [
    # ???????? ????????? ? ??????????? ????:
    'review_total_negative_word_counts', 'review_total_positive_word_counts',
    'review_total_word_counts_diff_norm', 'review_positive_word_proportion',

     # ????????, ????????? ? ???????? ????????????? ??????? ???????:
    'n_review_sentiments_neg', 'n_review_sentiments_neu',
    'n_review_sentiments_pos', 'n_review_sentiments_compound',
    'p_review_sentiments_neg', 'p_review_sentiments_neu',
    'p_review_sentiments_pos', 'p_review_sentiments_compound',
    'negative_polarity', 'positive_polarity', 
    'negative_review_pos', 'positive_review_pos', 
    'review_difference_with_top', 'afinn_score',

    # ????????? ??????????????:
    'total_number_of_reviews_reviewer_has_given', 'tag_count_nights'
]

cat_cols = [
    # ?????????????? ?????
    'average_score', 'additional_number_of_scoring', 'total_number_of_reviews',
    'hotel_code', 'hotel_country_code', 'district_code', 
    'district_rating', 'hotel_distance_to_center',

    # ?????????????? ??????? (?????):
    'tag_leisure_trip', 'tag_view_room', 'room_type_without Window', 
    'room_type_Suite', 'room_type_Standard', 'room_type_Double',
    'room_type_Superior', 'room_type_Club', 'room_type_Comfort', 
    'room_type_Deluxe', 'room_type_Premier', 'room_type_Classic', 
    'room_type_King', 'room_type_Single', 'room_type_Guestroom', 'room_type_Other',

    # ?????????????? ??????????:
    'reviewer_en', 'reviewer_n_est', 'reviewer_resident', 'reviewer_nationality_cat', 
    'tag_from_mobile', 'tag_with_pet',
    'tag_reviewer_type_1.0', 'tag_reviewer_type_2.0', 'tag_reviewer_type_3.0',
    'tag_reviewer_type_4.0', 'tag_reviewer_type_5.0', 'tag_reviewer_type_6.0',

    # ????? ???????:
    'review_week', 'review_month',

    # ?????????????? ??????:
    'review_classification', 'positive_words_more', 
    'negative_review_class', 'positive_review_class',
]

## 2.1 ?????? ????????????????????


????????? ????????? ????? ????? ?????? ?????????????? ?? ????????? ???????:

### ?????? ???????????????????? ??????????? ?????????

In [ ]:
def show_corr_heatmap(columns_list, title, method):
    """?????????? ?????????????? ?????????"""
    fig_, ax_ = plt.subplots(figsize=(15, 12))
    corr = hotels[columns_list].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, 
                annot=True, 
                linewidths=0.1, 
                ax=ax_, 
                mask=mask, 
                cmap='viridis',
                fmt='.1g')
    ax_.set_title(title, fontsize=18)
    plt.show()

In [ ]:
#?????? ????????? ??? ??????????? ?????????:
show_corr_heatmap(num_cols, '?????????? ??????????? ?????????', method='pearson');

???????? ?? ???????? ??? ???????? ?? ??? ??????? ?????? ?????????? ?????? 0.8.

In [ ]:
# ?????? ??????? ? ?????? ?? ????????:
drop_columns = ['n_review_sentiments_neu']

### ?????? ???????????????????? ?????????????? ?????????

???????? ?????????????? ???????? ?? 2 ????????? ??????:

In [ ]:
# ? ?????? ???? ??????? ?????????????? ????? ? ??????:
cat_cols_p1 = cat_cols[:24]

# ?? ?????? ???? - ?????????????? ??????????, ??????? ??????? ? ??????:
cat_cols_p2 = cat_cols[24:]

In [ ]:
# ?????? ????????? ??? ?????? ????? ?????????????? ?????????:
show_corr_heatmap(
    cat_cols_p1, 
    '?????????? ?????????????? ?????????\n(????????? ????? ? ????????? ??????)', 
    method='spearman'
);

In [ ]:
# ?????? ??????? ? ?????? ?? ????????:
drop_columns += ['additional_number_of_scoring']

In [ ]:
# ?????? ????????? ??? ?????? ????? ?????????????? ?????????:
show_corr_heatmap(
    cat_cols_p2, 
    '?????????? ?????????????? ?????????\n(????????? ??????????, ?????? ? ??????? ???????)', 
    method='spearman'
);

? ?? ????? ?????? ????? ??????? ??????? ?????? ??? ?????? (?????? ?????) ? ????????????? ?????? (?? ?????? ??????????? ? ?????????? ?? ??????? ???????).

In [ ]:
# ?????? ??????? ? ?????? ?? ????????:
drop_columns += ['review_month', 'review_classification']

In [ ]:
# ?????? ????????? ???????:
hotels_copy.drop(drop_columns, axis = 1, inplace=True)

print(f'??????? ???????????? ?????????? ?????????: {len(hotels_copy.columns)}')
print('(??????? ??????? "Sample" ? ??????? ???????)')

## 2.2 ?????? ?????????? ?????????

???????? ?????? ?????????? ?????????. ? ?? ??????? ??? ??????? ??????? ? ????????? - ?????????????? ??????????. ????????????? ??? ?????? ????????????? ???????? ANOVA. ?????? ?????? ???????? ?? ?????? ?? ?????????, ??? ????? ???????? ????????? ?????? ?? ???? ????????????:

In [ ]:
y = hotels_copy.query('sample == 1').drop(['sample'], axis=1).reviewer_score.values
X = hotels_copy.query('sample == 1').drop(['sample', 'reviewer_score'], axis=1)

# ????????????? ????????? ??????? ??????????:
imp_num = pd.Series(f_classif(X[X.columns], y)[0], index = X.columns)
imp_num.sort_values(inplace = True)

fig5, ax5 = plt.subplots(figsize=(15, 20))
imp_num.plot(kind = 'barh', color='green');

?????? ????????, ?? ??????????????? ??????? ?????????? ? ???????. ??? ???????? ????????? ????? ??????? ? ????? ???????????????, ? ????? ??????? ???????????????? ? ?????????.

???? ? ???? ???? ????????, ??? ????? ???? ?? ?????? ? ????? ????, ??? ??? ?? ?????? ????? ??????? ??????????? ? ????????? ?????????, ? ????????, ??????????? ??????? ????????? ?????????? ??????. ?? ???...)

In [ ]:
drop_columns = ['room_type_Other', 'room_type_Club', 'room_type_Comfort', 
                'room_type_Double', 'room_type_Premier', 'tag_with_pet',
                'tag_reviewer_type_3.0', 'tag_reviewer_type_4.0', 
                'tag_reviewer_type_5.0', 'tag_reviewer_type_6.0']

In [ ]:
# ?????? ?????????? ???????? ??????:
print(f'?????? ??? {len(drop_columns)} ???????? ? ????????? ???????')
hotels_copy.drop(drop_columns, axis = 1, inplace=True)

print(f'???????? ???????????? ?????????? ?????????: {len(hotels_copy.columns)}')
print('(??????? ??????? "Sample" ? ??????? ???????)')

# 3. ???????? ?????? ? ????????? ???????????? (Model Building):

### ????????? ????????? ?? ?????, ??????????? ??? ???????? ? ???????????? ??????:

In [ ]:
# ?????? ??????? ???????? ?????
train_data = hotels_copy.query('sample == 1').drop(['sample'], axis=1)
test_data = hotels_copy.query('sample == 0').drop(['sample'], axis=1)

y = train_data.reviewer_score.values
X = train_data.drop(['reviewer_score'], axis=1)

In [ ]:
# ????????????? ??????????? ??????? train_test_split ??? ?????????? ???????? ??????
# ??????? 20% ?????? ?? ????????? (???????? test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
# ????????? ???????:
test_data.shape, train_data.shape, X.shape, X_train.shape, X_test.shape

### ??????? ?????? ? ???????? ????????????:

In [ ]:
# ????????? ?? ???????
model = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

In [ ]:
%%time

# ??????? ?????? ?? ???????? ?????? ??????
model.fit(X_train, y_train)

# ????????????? ???????? ?????????? ? ?????????? y_pred
y_pred = model.predict(X_test)

In [ ]:
# ?????????? ????????????? ???????? (y_pred) ? ????????? (y_test), 
# ??????? Mean Absolute Percentage Error (MAPE) ?????????? ??????? ?????????? 
# ?????????? ?????? ????????????? ???????? ?? ???????????:

def mean_absolute_percentage_error(y_tr, y_pr):
    """????????? ??????? ?????????? ??????"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 2))

### ?????? ????? ????????? ????????? ? ?????????? ??????:

In [ ]:
# ? RandomForestRegressor ???? ??????????? ??????? ????? ?????? ????????
plt.rcParams['figure.figsize'] = (10,10)
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(15).plot(kind='barh', color='green');

### ???????? ???????????? ??????? ??????????:

In [ ]:
test_data = test_data.drop(['reviewer_score'], axis=1)
predict_submission = model.predict(test_data)
sample_submission['reviewer_score'] = predict_submission

#  ????????? ?????????:
sample_submission.to_csv('submission.csv', index=False)
sample_submission.head(10)

#### ????? 15 ??????? ??? ?????????? ?????? ????????? ? ???????? MAPE 11.92 - ?????? ??? ???????? - ??????? ????? ? ????? ?????????? ??????? ?? ????? (????????????? ????????????? ???? ???????). ????????? - ??? ????????, ????????????????? ? ???????? ???????????? ??????. ????????, ??? ???????? MAPE ?? ?????????????? ?????? - 14.14%, ?????? ??? ?????? ???????????? EDA ??? ??????? ???????? ??? ???????? ?? 15,5 ?????????. ? ???? ????????? ????????? ???? ? ????????? ?? ?????? ? ??????????? ?????????? ???? ? ?????? ?????? ??????. ?? ???????, ??????? ???? ??????? ????????, ?????????? ?? ??????????? ????????????? ??????? ??????.

P.S. ??? ??????? ??????????????, ?? ?????? ??????? ????? ???????. ? ????????? ????????, ?? ???????? ?? ????? ???? ? ???????? ??????, ?????? ?? ???? EDA (?????? ? ??????????) ??? ??????? ???????? ???????? ??????? MAPE ?? ???????? 11.55. ????????, ??? ??? ??????? ?????? ????? ????????????, ??????, ??? ???????????? ??? ????? ??????? ?? ????????? :D

???? ?? ? ?????? ?????? ?????????? ???? ?????? - ????? ??? ??????????? ? ?????. ?? ??? ??????? ?????? ? ?????????????!